In [ ]:

import os
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, jaccard_score
import numpy as np
import torchvision.transforms.functional as TF
import torchvision.transforms as transforms
import random
import csv
import time

In [ ]:

# Carregando o Dataset Original
class SegmentationDatasetOriginal(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        mask = (mask > 0).float()

        return image, mask

In [ ]:

# Dataset com filtro de máscaras pretas
class SegmentationDatasetFiltrado(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.transform = transform
        self.image_paths = []
        self.mask_paths = []

        for img_path, mask_path in zip(image_paths, mask_paths):
            mask = Image.open(mask_path).convert("L")
            if np.array(mask).sum() > 0:
                self.image_paths.append(img_path)
                self.mask_paths.append(mask_path)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        mask = (mask > 0).float()
        return image, mask

In [ ]:
# Carregando o Dataset Filtrado e augmentado
class SegmentationDatasetAugumentado(Dataset):
    def __init__(self, image_paths, mask_paths, augment=False):
        self.augment = augment # Flag para ativar o augmentation
        self.image_paths = []
        self.mask_paths = []

        # Filtrar máscaras inteiramente pretas
        for img_path, mask_path in zip(image_paths, mask_paths):
            mask = Image.open(mask_path).convert("L")
            if np.array(mask).sum() > 0: 
                self.image_paths.append(img_path)
                self.mask_paths.append(mask_path)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        # DATA AUGMENTATION
        if self.augment:
           
            # Flip Horizontal Aleatório (50% de chance)
            if random.random() > 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)
                
            # Flip Vertical Aleatório (50% de chance)
            if random.random() > 0.5:
                image = TF.vflip(image)
                mask = TF.vflip(mask)
                
            # Rotação Aleatória (entre -30 e 30 graus)
            if random.random() > 0.5:
                angle = random.uniform(-30, 30)
                image = TF.rotate(image, angle)
                mask = TF.rotate(mask, angle)
           
            # Ajuste de brilho e contraste (50% de chance)
            if random.random() > 0.5:
                color_jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2)
                image = color_jitter(image)

        # CONVERSÃO PARA TENSOR
        image = TF.to_tensor(image)
        mask = TF.to_tensor(mask)

        # Binarização da máscara (garantir que seja 0 ou 1)
        mask = (mask > 0).float()
        
        return image, mask

In [ ]:
# Função para preparar os dados
def prepararDados(SegmentationDataset, seed, augment):

    image_dir = "tiles/image"
    mask_dir = "tiles/mask"

    image_paths = sorted([os.path.join(image_dir, f) for f in os.listdir(image_dir)])
    mask_paths = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir)])

    train_val_imgs, test_imgs, train_val_masks, test_masks = train_test_split(image_paths, mask_paths, test_size=0.2, random_state=seed)
    train_imgs, val_imgs, train_masks, val_masks = train_test_split(train_val_imgs, train_val_masks, test_size=0.2, random_state=seed)

    transform = transforms.ToTensor()

    # Com Augmentation
    if augment == True:
        train_dataset = SegmentationDataset(train_imgs, train_masks, augment=True)
        val_dataset = SegmentationDataset(val_imgs, val_masks, augment=False)
        test_dataset = SegmentationDataset(test_imgs, test_masks, augment=False)
    
    # Sem Augmentation
    else:    
        train_dataset = SegmentationDataset(train_imgs, train_masks, transform=transform)
        val_dataset = SegmentationDataset(val_imgs, val_masks, transform=transform)
        test_dataset = SegmentationDataset(test_imgs, test_masks, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    print(f"Imagens de treino: {len(train_dataset)}")
    print(f"Imagens de validação: {len(val_dataset)}")
    print(f"Imagens de teste: {len(test_dataset)}")

    # Visualização de amostras
    plt.figure(figsize=(4, 30))
    for i in range(3):
        img, mask = train_dataset[i]
        plt.subplot(10, 2, 2*i+1)
        plt.imshow(img.permute(1, 2, 0))
        plt.axis('off')
        plt.subplot(10, 2, 2*i+2)
        plt.imshow(mask.squeeze(), cmap='gray')
        plt.axis('off')
    plt.show()

    return train_loader, val_loader, test_loader

In [ ]:

# Modelo SegNet
class SegNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(SegNet, self).__init__()

        self.enc1 = nn.Sequential(nn.Conv2d(in_channels, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.pool1 = nn.MaxPool2d(2, stride=2, return_indices=True)

        self.enc2 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        self.pool2 = nn.MaxPool2d(2, stride=2, return_indices=True)

        self.unpool2 = nn.MaxUnpool2d(2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())

        self.unpool1 = nn.MaxUnpool2d(2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(64, out_channels, 3, padding=1))

    def forward(self, x):
        x1 = self.enc1(x)
        x1p, idx1 = self.pool1(x1)

        x2 = self.enc2(x1p)
        x2p, idx2 = self.pool2(x2)

        x2u = self.unpool2(x2p, idx2, output_size=x2.size())
        x2d = self.dec2(x2u)

        x1u = self.unpool1(x2d, idx1, output_size=x1.size())
        x1d = self.dec1(x1u)

        return torch.sigmoid(x1d)

In [ ]:

# Função para treinar modelo
def treinarModelo(device, train_loader, val_loader):

    model = SegNet().to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCELoss()

    n_epochs = 100
    train_losses, val_losses = [], []

    # ==========================================
    # CONFIGURAÇÕES DOS "CALLBACKS" MANUAIS
    # ==========================================
    patience = 15
    epochs_no_improve = 0
    best_val_loss = float('inf')
    model_file = "melhor_segnet.pth"
    history_file = "history_segnet.csv"

    # Inicializar o arquivo CSV com os cabeçalhos
    with open(history_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'train_loss', 'val_loss'])

    print(f" @ Iniciando Treinamento da Segnet\n")

    start_time = time.time()

    for epoch in range(n_epochs):
        model.train()
        train_loss = 0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, masks)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, masks)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        print(f"Epoch {epoch+1}/{n_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

        # ==========================================
        # LÓGICA DE CALLBACKS (CSV, Checkpoint, Early Stopping)
        # ==========================================
        
        # 1. CSV Logger
        with open(history_file, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([epoch + 1, train_loss, val_loss])

        # 2. Model Checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_epoch = epoch + 1
            torch.save(model.state_dict(), model_file)
            print(" -> Val loss melhorou! Modelo salvo.")
        else:
            epochs_no_improve += 1
            print(f" -> Sem melhora ({epochs_no_improve}/{patience})")

        # 3. Early Stopping
        if epochs_no_improve >= patience:
            print(f"\n[!] Early stopping acionado na época {epoch+1}! O treino foi interrompido.")
            break

    end_time = time.time()
    tempo_de_execucao = end_time - start_time
    total_epocas = epoch + 1

    # 4. Restore Best Weights (Carregar os melhores pesos antes do teste)
    print("\nRestaurando os melhores pesos para a fase de testes...")
    model.load_state_dict(torch.load(model_file))

    return model, train_losses, val_losses, tempo_de_execucao, total_epocas, best_epoch


In [ ]:
# Plot do gráfico das curvas
def plotCurvas(train_losses, val_losses):

    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:

# Avaliação com IoU e F1
def avaliarModelo(device, model, test_loader):

    model.eval()
    iou_scores = []
    f1_scores = []

    for i, (imgs, masks) in enumerate(test_loader):
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = (outputs > 0.5).float().cpu().numpy().squeeze().astype(np.uint8)
        true = masks.numpy().squeeze().astype(np.uint8)

        if np.sum(true) == 0 and np.sum(preds) == 0:
            continue

        iou = jaccard_score(true.flatten(), preds.flatten(), zero_division=1)
        f1 = f1_score(true.flatten(), preds.flatten(), zero_division=1)

        iou_scores.append(iou)
        f1_scores.append(f1)

        if i < 2:
            plt.figure(figsize=(16, 4))
            plt.subplot(1, 4, 1)
            plt.imshow(imgs.cpu().squeeze().permute(1, 2, 0))
            plt.title("Imagem Original")
            plt.axis('off')

            plt.subplot(1, 4, 2)
            plt.imshow(true, cmap='gray')
            plt.title("Máscara")
            plt.axis('off')

            plt.subplot(1, 4, 3)
            plt.imshow(preds, cmap='gray')
            plt.title("Predição")
            plt.axis('off')

            plt.subplot(1, 4, 4)
            plt.imshow(true, cmap='gray', alpha=0.5)
            plt.imshow(preds, cmap='jet', alpha=0.5)
            plt.title("Sobreposição")
            plt.axis('off')
            plt.show()

    return iou_scores, f1_scores


In [ ]:
# Salvar os resultados
arquivo_resultados = "resultados_bateria_segnet.csv"
nome_configuracao = "augumentado" 

# Criar o arquivo e escrever o cabeçalho
with open(arquivo_resultados, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Configuracao', 'Seed', 'Rodada', 'Tempo_Segundos', 'Total_Epocas', 'Melhor_Epoca', 'IoU_Medio', 'F1_Medio'])

# Bateria de testes
seeds = [16, 32, 64, 128]
rodadas = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for seed in seeds:

    for rodada in range(rodadas):
    
        print(f"Seed: {seed} | Rodada {rodada}")

        # Travar gerador de números aleatórios
        random.seed(seed)

        # Carregar o dataset SegmentationDatasetOriginal | SegmentationDatasetFiltrado | SegmentationDatasetAugumentado     
        train_loader, val_loader, test_loader = prepararDados(SegmentationDatasetAugumentado, seed, True)

        model, train_losses, val_losses, tempo_de_execucao, total_epocas, best_epoch = treinarModelo(device, train_loader, val_loader)

        print(f"Tempo de execução: {tempo_de_execucao}")
        print(f"Total de épocas: {total_epocas}")
        print(f"Melhor época: {best_epoch}")

        plotCurvas(train_losses, val_losses)

        iou_scores, f1_scores = avaliarModelo(device, model, test_loader)

        print(f"IoU médio: {np.mean(iou_scores):.4f}")
        print(f"F1-score médio: {np.mean(f1_scores):.4f}\n\n")

        with open(arquivo_resultados, mode='a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                nome_configuracao, 
                seed, 
                rodada + 1, 
                f"{tempo_de_execucao:.2f}", 
                total_epocas, 
                best_epoch, 
                f"{np.mean(iou_scores):.4f}", 
                f"{np.mean(f1_scores):.4f}"
            ])